# Baseline retrieval comparison

Compare the three retrieval baselines on exactly the same registered BioASQ datasets and evaluation definition:

1. **BM25**
2. **`sentence-transformers/all-MiniLM-L6-v2`** with standard cosine similarity
3. **`Qwen/Qwen3-Embedding-0.6B`** with cosine similarity and a biomedical query instruction

No embedding-space post-processing is applied in this notebook. Dense baselines therefore use the identity embedding transform.


In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/lohex/retrieval-benchlab.git"
REPO_BRANCH = "main"
REPO_ROOT = Path("/content/retrieval-benchlab")

if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}
    else:
        !git -C {REPO_ROOT} fetch origin {REPO_BRANCH}
        !git -C {REPO_ROOT} checkout {REPO_BRANCH}
        !git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets sentence-transformers rank-bm25


In [ ]:
import logging
import sqlite3

import pandas as pd
import torch
from IPython.display import display

from src.evaluate import RuntimeConfig, evaluate, register_evaluation, register_pipeline
from src.io import mount_google_drive

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logger = logging.getLogger("retrieval-baselines")
mount_google_drive()


## Shared evaluation configuration

Google Drive is mounted before any dataset or SQLite path under `/content/drive/MyDrive` is accessed.

The current persistence schema keys results by `(pipeline_id, evaluation_id, dataset_id)`. Databases produced by the earlier schema are intentionally not migrated. Set `RESET_DATABASES = True` once to delete the old registry/results databases and recreate them with the current schema. Set it back to `False` afterwards.


In [ ]:
DATASETS_ROOT = "/content/drive/MyDrive/Retreaval/data"
REGISTRY_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/datasets.sqlite"
RESULTS_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/results.sqlite"
RESET_DATABASES = False

def delete_sqlite_database(path: str) -> None:
    database_path = Path(path)
    for suffix in ("", "-wal", "-shm"):
        Path(f"{database_path}{suffix}").unlink(missing_ok=True)

def assert_current_results_schema(path: str) -> None:
    database_path = Path(path)
    if not database_path.exists():
        return
    with sqlite3.connect(database_path) as connection:
        table = connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name='evaluation_runs'"
        ).fetchone()
        if table is None:
            return
        columns = {
            str(row[1])
            for row in connection.execute("PRAGMA table_info(evaluation_runs)")
        }
    if "evaluation_id" not in columns:
        raise RuntimeError(
            "results.sqlite uses the pre-evaluation_id schema. "
            "Set RESET_DATABASES = True, rerun this cell once, then set it back to False."
        )

if RESET_DATABASES:
    delete_sqlite_database(REGISTRY_DB_PATH)
    delete_sqlite_database(RESULTS_DB_PATH)
    logger.warning("Deleted evaluation registry and results databases")
else:
    assert_current_results_schema(RESULTS_DB_PATH)

METRIC_CONFIG = {
    "mrr_at_k": (10,),
    "ndcg_at_k": (10,),
    "accuracy_at_k": (1, 3, 5, 10, 100),
    "precision_recall_at_k": (1, 3, 5, 10, 100),
    "map_at_k": (100,),
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_RUNTIME = RuntimeConfig(
    batch_size=64,
    corpus_scan_size=10_000,
    show_progress_bar=True,
    device=DEVICE,
)
QWEN_RUNTIME = RuntimeConfig(
    batch_size=8,
    corpus_scan_size=10_000,
    show_progress_bar=True,
    device=DEVICE,
)
evaluation_id = register_evaluation(METRIC_CONFIG, registry_db_path=REGISTRY_DB_PATH)
logger.info("Evaluation: %s", evaluation_id)
logger.info("Device: %s", DEVICE)


## Register the three baselines

All dense pipelines use the default identity transform. Qwen is capped at **512 tokens** per query/document. The cap is stored in `model_kwargs`, so it is ranking-relevant and therefore changes pipeline identity. This avoids pathological memory use from the model's much larger native context window when encoding batches of abstracts in Colab.

The biomedical Qwen instruction is prepended explicitly by the retriever. When an explicit `query_prompt` is configured, the retriever uses plain `encode()` rather than `encode_query()` so the model's built-in query prompt is not applied a second time.


In [ ]:
bm25_pipeline_id = register_pipeline(
    retriever_type="bm25",
    registry_db_path=REGISTRY_DB_PATH,
)

minilm_pipeline_id = register_pipeline(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    similarity_metric="cosine",
    registry_db_path=REGISTRY_DB_PATH,
)

qwen_pipeline_id = register_pipeline(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    similarity_metric="cosine",
    model_kwargs={"max_seq_length": 512},
    query_prompt=(
        "Instruct: Given a biomedical question, retrieve relevant scientific "
        "passages that answer the question\nQuery: "
    ),
    registry_db_path=REGISTRY_DB_PATH,
)

pipelines = {
    "BM25": (bm25_pipeline_id, DEFAULT_RUNTIME),
    "all-MiniLM-L6-v2": (minilm_pipeline_id, DEFAULT_RUNTIME),
    "Qwen3-Embedding-0.6B": (qwen_pipeline_id, QWEN_RUNTIME),
}
{name: pipeline_id for name, (pipeline_id, _) in pipelines.items()}


## Evaluate every latest dataset version

Result identity is `(pipeline_id, evaluation_id, dataset_id)`. Existing results for the same triple are loaded rather than recomputed. Runtime batch size is deliberately allowed to differ by backend because it cannot change rankings.


In [ ]:
records = []
for pipeline_name, (pipeline_id, runtime) in pipelines.items():
    logger.info("Evaluating %s with batch_size=%d", pipeline_name, runtime.batch_size)
    outcomes = evaluate(
        pipeline_id=pipeline_id,
        evaluation_id=evaluation_id,
        datasets_root=DATASETS_ROOT,
        runtime=runtime,
        registry_db_path=REGISTRY_DB_PATH,
        results_db_path=RESULTS_DB_PATH,
    )
    for outcome in outcomes:
        record = {
            "pipeline": pipeline_name,
            "dataset": outcome.dataset_name,
            "version": outcome.dataset_version,
            "status": outcome.status.value,
            "dataset_id": outcome.dataset_id,
            "result_id": outcome.result_id,
        }
        if outcome.metrics is not None:
            record.update(outcome.metrics)
        records.append(record)

results_table = pd.DataFrame(records).sort_values(["pipeline", "dataset", "version"])
identifier_columns = {"pipeline", "dataset", "version", "status", "dataset_id", "result_id"}
metric_columns = [column for column in results_table.columns if column not in identifier_columns]
display(results_table.style.format({column: "{:.4f}" for column in metric_columns}, na_rep=""))
